In [ ]:
import os
import keras
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope
from hgq.layers import QDense, QSoftmax
from hgq.utils.sugar import FreeEBOPs, PBar

2026-04-23 15:22:26.671263: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776950546.688358   17679 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776950546.693015   17679 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-23 15:22:26.713095: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets, can also be binarized instead
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
#X_train, X_test = (X_train > 127).astype(np.float32), (X_test > 127).astype(np.float32)

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
y_val = keras.utils.to_categorical(y_val, 10)

dataset_train = Dataset(X_train, y_train, batch_size=2048, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=2048, device='gpu:0')
dataset_test = Dataset(X_test, y_test, batch_size=2048, device='gpu:0')

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1776950550.813985   17679 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
import keras
from hgq.layers import QDense, QConv2D
from hgq.config import LayerConfigScope, QuantizerConfigScope

with (
      QuantizerConfigScope(place='all', default_q_type='kbi', f0=5, i0=2, overflow_mode='SAT_SYM', trainable=True),
      #QuantizerConfigScope(place='bias', default_q_type='kif', f0=5, i0=2, overflow_mode='WRAP', trainable=True),
      QuantizerConfigScope(place='datalane', default_q_type='kif', f0=5, i0=2, heterogeneous_axis=None, homogeneous_axis=(0, 1)),
      LayerConfigScope(enable_ebops=True, beta0=1e-7),
   ):
      inputs = keras.Input(shape=input_shape)

      x = QConv2D(16, (3, 3), activation='relu')(inputs)
      x = keras.layers.MaxPooling2D((2, 2))(x)
      
      x = QConv2D(32, (3, 3), activation='relu')(x)
      x = keras.layers.MaxPooling2D((2, 2))(x)

      x = keras.layers.Flatten()(x)
      x = keras.layers.Dropout(0.4)(x)

      outputs = QDense(10)(x)

      model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
    jit_compile=True,
    steps_per_execution=10
)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d (QConv2D)              │ (None, 26, 26, 16)     │           726 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_1 (QConv2D)            │ (None, 11, 11, 32)     │        19,186 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 10)             │        32,045 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,957 (164.73 KB)

 Trainable params: 38,904 (151.97 KB)

 Non-trainable params: 13,053 (12.76 KB)

In [ ]:
import keras
import numpy as np
from math import cos, pi
from hgq.utils.sugar import BetaScheduler, Dataset, FreeEBOPs, ParetoFront, PBar, PieceWiseSchedule, EarlyStoppingWithEbopsThres
from keras.callbacks import LearningRateScheduler, CSVLogger

OUTPUT_PATH = '/home/slopin/DAT255-project/Modeller/model_HGQ/model_outputs_2'

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)
    print(f"Created new folder: {OUTPUT_PATH}")

def cosine_decay_restarts_schedule(
    initial_lr, first_decay_steps, m_mul, alpha
):
    def schedule(step):
        cycle = step // first_decay_steps
        x = (step % first_decay_steps) / first_decay_steps
        lr = alpha + 0.5 * (initial_lr * (m_mul ** cycle) - alpha) * (1 + cos(pi * x))
        return lr
    return schedule
def warmup_cosine_schedule(
    initial_lr,
    warmup_steps,
    first_decay_steps,
    m_mul=0.9,
    alpha=1e-5
):
    cosine = cosine_decay_restarts_schedule(
        initial_lr,
        first_decay_steps,
        m_mul=m_mul,
        alpha=alpha
    )

    def schedule(step):
        # Warmup phase
        if step < warmup_steps:
            return initial_lr * (0.1 + 0.9 * step / warmup_steps)

        # Cosine phase
        return cosine(step - warmup_steps)

    return schedule

pbar = PBar(
        'loss: {loss:.2f}/{val_loss:.2f} - acc: {accuracy:.4f}/{val_accuracy:.4f} - lr: {learning_rate:.2e} - beta: {beta:.1e}'
    )
ebops = FreeEBOPs()
pareto = ParetoFront(
        OUTPUT_PATH,
        ['val_accuracy', 'ebops'],
        [1, -1],
        fname_format='epoch={epoch}-val_acc={val_accuracy:.3f}-ebops={ebops}-val_loss={val_loss:.3f}.keras',
    )
estop = EarlyStoppingWithEbopsThres(ebops_threshold=3000, monitor='val_accuracy', patience=30, restore_best_weights=True)
beta_sched = BetaScheduler(
    PieceWiseSchedule([
        (0,    0.0, 'constant'),
        (50,   1e-7, 'log'),
        (500,  1e-6, 'log'),
        (5000, 1e-5, 'constant'),
        (6000, 1e-5, 'log'),
        (8000, 1e-4, 'constant'),
    ])
)
lr_sched = LearningRateScheduler(
    warmup_cosine_schedule(
        initial_lr=1e-3,
        warmup_steps=50,
        first_decay_steps=250,
        m_mul=0.9,
        alpha=5e-4
    )
)

csv_logger = CSVLogger('/home/slopin/DAT255-project/Modeller/model_HGQ/MNIST_HGQ_DynamicTraining_log.csv', append=True, separator=';')

In [ ]:
callbacks = [ebops, lr_sched, beta_sched, pbar, pareto, estop, csv_logger]

In [ ]:
model.fit(dataset_train, epochs=8000, validation_data=dataset_val, callbacks=callbacks, verbose=0)

  0%|          | 0/8000 [00:00<?, ?epoch/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1776950614.648061   17774 service.cc:148] XLA service 0x7f9f80005a40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776950614.648105   17774 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-23 15:23:34.777403: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776950615.251492   17774 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-04-23 15:23:36.342212: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2064', 48 bytes spill stores, 48 bytes spill loads

2026-04-23 15:23:36.434238: I external/local_xla/xla/stre

KeyboardInterrupt: 

In [ ]:
#pull model, search for best model can be automated, but is done manually in this case.
#Aiming for the lowest EBOPs with the highest accuracy.
from keras.models import load_model

model = load_model('/home/slopin/DAT255-project/Modeller/model_HGQ/model_outputs_StaticTraining_CNN/epoch=4468-val_acc=0.965-ebops=34215-val_loss=0.128.keras')

score = model.evaluate(dataset_test)
model.save('/home/slopin/DAT255-project/Modeller/model_HGQ/MNIST_CNN_HGQ_StaticTraining_model.keras')
print("Test loss:", score[0])
print("Test accuracy:", score[1])